In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import json

# ============ 1. 定义工具（使用 @tool 装饰器） ============

@tool
def get_weather(location: str) -> str:
    """
    获取指定地点的天气信息
    
    Args:
        location: 地点名称，比如：北京、上海
    
    Returns:
        该地点的天气描述
    """
    # 模拟天气数据
    weather_data = {
        "北京": "晴朗，气温25°C，空气质量优",
        "上海": "多云，气温28°C，适合户外活动",
        "广州": "小雨，气温30°C，记得带伞",
        "深圳": "雷阵雨，气温29°C，注意安全",
        "成都": "阴天，气温22°C，凉爽舒适",
        "杭州": "晴天，气温26°C，西湖美景"
    }
    
    # 如果查询的地点在我们预定义的数据中
    if location in weather_data:
        return f"{location}天气：{weather_data[location]}"
    else:
        # 随机返回一个默认天气
        return f"{location}天气：晴朗，气温24°C（模拟数据）"

@tool
def get_current_time(location: str) -> str:
    """
    获取指定地点的当前时间
    
    Args:
        location: 地点名称，比如：北京、纽约
    
    Returns:
        该地点的时间信息
    """
    import datetime
    
    # 模拟不同时区的时间（实际应用中可以调用时间API）
    time_data = {
        "北京": datetime.datetime.now().strftime("%H:%M"),
        "上海": datetime.datetime.now().strftime("%H:%M"),
        "纽约": (datetime.datetime.now() - datetime.timedelta(hours=12)).strftime("%H:%M"),
        "伦敦": (datetime.datetime.now() - datetime.timedelta(hours=7)).strftime("%H:%M"),
        "东京": (datetime.datetime.now() + datetime.timedelta(hours=1)).strftime("%H:%M")
    }
    
    if location in time_data:
        return f"{location}当前时间：{time_data[location]}"
    else:
        return f"{location}当前时间：{datetime.datetime.now().strftime('%H:%M')}（使用本地时间）"

@tool
def calculate_temperature_average(city1: str, city2: str) -> str:
    """
    计算两个城市的平均温度
    
    Args:
        city1: 第一个城市名称
        city2: 第二个城市名称
    
    Returns:
        两个城市的平均温度
    """
    # 模拟温度数据
    temps = {
        "北京": 25,
        "上海": 28,
        "广州": 30,
        "深圳": 29,
        "成都": 22,
        "杭州": 26
    }
    
    temp1 = temps.get(city1, 24)
    temp2 = temps.get(city2, 24)
    avg = (temp1 + temp2) / 2
    
    return f"{city1}和{city2}的平均气温是{avg:.1f}°C"

# ============ 2. 创建工具列表 ============

tools = [get_weather, get_current_time, calculate_temperature_average]

# ============ 3. 初始化大模型并绑定工具 ============

def create_chat_model():
    """
    创建并配置聊天模型
    """
    # 初始化模型
    model = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,  # 设置为0让输出更稳定
        api_key="你的API密钥",  # 替换成你的API密钥
        base_url="https://api.openai.com/v1"
    )
    
    # 关键步骤：使用 bind_tools 绑定工具
    model_with_tools = model.bind_tools(tools)
    
    return model_with_tools

# ============ 4. 创建工具执行函数 ============

def execute_tool_calls(tool_calls):
    """
    执行模型要求的工具调用
    
    Args:
        tool_calls: 模型返回的工具调用信息
    
    Returns:
        工具执行结果的列表
    """
    results = []
    
    # 创建工具名称到实际工具的映射
    tools_dict = {tool.name: tool for tool in tools}
    
    for tool_call in tool_calls:
        # 获取工具名称和参数
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"  执行工具: {tool_name}")
        print(f"  参数: {tool_args}")
        
        # 调用对应的工具
        if tool_name in tools_dict:
            result = tools_dict[tool_name].invoke(tool_args)
            print(f"  结果: {result}")
            
            results.append({
                "tool_call_id": tool_call["id"],
                "result": result
            })
        else:
            print(f"  错误: 未找到工具 {tool_name}")
    
    return results

# ============ 5. 主对话函数 ============

def chat_with_tools(user_input):
    """
    与大模型对话，自动处理工具调用
    """
    print(f"\n👤 用户: {user_input}")
    print("-" * 50)
    
    # 创建绑定了工具的模型
    model = create_chat_model()
    
    # 初始化消息列表
    messages = [
        {"role": "system", "content": "你是一个有用的助手，可以根据用户的问题调用相应的工具。调用工具后，请用友好的语气告诉用户结果。"},
        {"role": "user", "content": user_input}
    ]
    
    # 第一步：调用模型
    print("🤔 模型思考中...")
    response = model.invoke(messages)
    
    # 检查是否需要调用工具
    if hasattr(response, "tool_calls") and response.tool_calls:
        print("🔧 需要调用工具！")
        
        # 将模型的回复添加到消息历史
        messages.append(response)
        
        # 执行工具调用
        tool_results = execute_tool_calls(response.tool_calls)
        
        # 将工具结果添加到消息历史
        for result in tool_results:
            messages.append({
                "role": "tool",
                "content": result["result"],
                "tool_call_id": result["tool_call_id"]
            })
        
        # 第二步：让模型根据工具结果生成最终回答
        print("🤔 模型整合工具结果中...")
        final_response = model.invoke(messages)
        
        return final_response.content
    else:
        # 不需要调用工具，直接返回
        return response.content

# ============ 6. 简化的本地测试版本（不需要API密钥） ============

def simple_test_without_api():
    """
    不需要API密钥的简化测试版本
    模拟 bind_tools 的工作流程
    """
    print("=" * 60)
    print("简化版 bind_tools 演示（模拟运行）")
    print("=" * 60)
    
    # 模拟对话场景
    test_queries = [
        "北京天气怎么样？",
        "上海现在几点了？", 
        "北京和上海的平均温度是多少？",
        "我想去广州旅游，那边天气如何？",
        "帮我算算北京和广州的平均气温"
    ]
    
    tools_dict = {tool.name: tool for tool in tools}
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n📝 测试 {i}: {query}")
        print("-" * 40)
        
        # 模拟模型识别意图（这部分在实际中由大模型完成）
        if "天气" in query and "平均" not in query:
            # 提取城市
            if "北京" in query:
                tool_name = "get_weather"
                args = {"location": "北京"}
            elif "上海" in query:
                tool_name = "get_weather" 
                args = {"location": "上海"}
            elif "广州" in query:
                tool_name = "get_weather"
                args = {"location": "广州"}
            else:
                tool_name = "get_weather"
                args = {"location": "北京"}
            
            print(f"🔧 bind_tools 触发调用: {tool_name}")
            print(f"📊 参数: {args}")
            
            # 调用工具
            result = tools_dict[tool_name].invoke(args)
            print(f"📝 工具返回: {result}")
            print(f"💬 最终回答: 我帮你查了，{result}。祝你有美好的一天！")
            
        elif "时间" in query:
            if "上海" in query:
                args = {"location": "上海"}
            elif "纽约" in query:
                args = {"location": "纽约"}
            else:
                args = {"location": "北京"}
            
            print(f"🔧 bind_tools 触发调用: get_current_time")
            print(f"📊 参数: {args}")
            
            result = tools_dict["get_current_time"].invoke(args)
            print(f"📝 工具返回: {result}")
            print(f"💬 最终回答: {result}，需要我帮你查其他城市的时间吗？")
            
        elif "平均" in query:
            if "北京" in query and "上海" in query:
                args = {"city1": "北京", "city2": "上海"}
            elif "北京" in query and "广州" in query:
                args = {"city1": "北京", "city2": "广州"}
            else:
                args = {"city1": "北京", "city2": "上海"}
            
            print(f"🔧 bind_tools 触发调用: calculate_temperature_average")
            print(f"📊 参数: {args}")
            
            result = tools_dict["calculate_temperature_average"].invoke(args)
            print(f"📝 工具返回: {result}")
            print(f"💬 最终回答: 计算完成！{result}。")

# ============ 7. 主程序入口 ============

if __name__ == "__main__":
    print("=" * 60)
    print("LangChain bind_tools 完整示例")
    print("=" * 60)
    print("\n📌 什么是 bind_tools？")
    print("  bind_tools 是 LangChain 的功能，让大模型可以自动调用工具")
    print("  它处理了工具描述的格式转换，让代码更简洁")
    
    print("\n📋 可用的工具:")
    for tool in tools:
        print(f"  • {tool.name}: {tool.description}")
    
    print("\n" + "=" * 60)
    print("选项:")
    print("1. 运行简化测试版 (不需要API密钥)")
    print("2. 运行完整版 (需要OpenAI API密钥)")
    print("=" * 60)
    
    choice = input("\n请选择 (1 或 2): ").strip()
    
    if choice == "1":
        simple_test_without_api()
        
    elif choice == "2":
        print("\n⚠️  运行完整版需要有效的OpenAI API密钥")
        api_key = input("请输入你的OpenAI API密钥: ").strip()
        
        if api_key:
            try:
                # 这里应该使用你输入的API密钥
                print("\n🌟 开始对话 (输入'退出'结束)")
                while True:
                    user_input = input("\n请输入你的问题: ")
                    if user_input.lower() in ['退出', 'exit', 'quit']:
                        print("再见！")
                        break
                    
                    response = chat_with_tools(user_input)
                    print(f"🤖 助手: {response}")
                    
            except Exception as e:
                print(f"❌ 错误: {e}")
                print("请检查你的API密钥是否正确")
        else:
            print("未输入API密钥，运行简化版...")
            simple_test_without_api()
    else:
        print("无效选择，运行简化版...")
        simple_test_without_api()